In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 
from delta.tables import DeltaTable

In [0]:
order_header = spark.read.table("databricksintech.silver.order_header")
order_header.limit(3).display()

Sales_order_id,Revision_number,Order_date,Due_date,Ship_date,Status,Online_order_flag,Sales_order_number,Purchase_order_number,Account_number,Customer_id,Ship_to_address_id,Bill_to_address_id,Ship_method,Credit_card_approval_code,Sub_total,Tax_amount,Freight,Total_due,Modified_date
71774,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71774,PO348186287,10-4020-000609,29847,1092,1092,CARGO TRANSPORT 5,null,880.3484,70.4279,22.0087,972.785,2008-06-08
71776,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71776,PO19952192051,10-4020-000106,30072,640,640,CARGO TRANSPORT 5,null,78.81,6.3048,1.9703,87.0851,2008-06-08
71780,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71780,PO19604173239,10-4020-000340,30113,653,653,CARGO TRANSPORT 5,null,38418.69,3073.495,960.4672,42452.652,2008-06-08


In [0]:
order_detail = spark.read.table("databricksintech.silver.order_detail")
order_detail.limit(3).display()

Sales_order_id,Sales_order_detail_id,Order_qty,Product_id,Unit_price,Unit_price_discount,Line_total,Modified_date
71774,110562,1.0,836,356.898,0.0,356.898,2008-06-01
71774,110563,1.0,822,356.898,0.0,356.898,2008-06-01
71776,110567,1.0,907,63.9,0.0,63.9,2008-06-01


In [0]:
order_detail = order_detail.withColumnRenamed('Sales_order_id', "Sales_order_id2")
order_detail = order_detail.withColumn('Total_order_price', col('Order_qty') * col("Unit_price") * (1 - col("Unit_price_discount")))
order_detail = order_detail.groupby("Sales_order_id2").agg(sum("Total_order_price").alias("Total_order_price"), sum("Unit_price_discount").alias("Unit_price_discount"))

In [0]:
orders = order_header.join(order_detail, order_header.Sales_order_id == order_detail.Sales_order_id2, "inner")
orders = orders.withColumn("Sub_total", col("Total_order_price"))\
               .withColumn("Total_due", col("Sub_total") + col("Tax_amount") + col("Freight"))

orders = orders.drop("Total_order_price", "Sales_order_id2")
orders.limit(10).display()

Sales_order_id,Revision_number,Order_date,Due_date,Ship_date,Status,Online_order_flag,Sales_order_number,Purchase_order_number,Account_number,Customer_id,Ship_to_address_id,Bill_to_address_id,Ship_method,Credit_card_approval_code,Sub_total,Tax_amount,Freight,Total_due,Modified_date,Unit_price_discount
71774,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71774,PO348186287,10-4020-000609,29847,1092,1092,CARGO TRANSPORT 5,null,713.7960205078125,70.4279,22.0087,806.2326221466064,2008-06-08,0.0
71776,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71776,PO19952192051,10-4020-000106,30072,640,640,CARGO TRANSPORT 5,null,63.900001525878906,6.3048,1.9703,72.17510151863098,2008-06-08,0.0
71780,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71780,PO19604173239,10-4020-000340,30113,653,653,CARGO TRANSPORT 5,null,29923.00828152317,3073.495,960.4672,33956.97062283177,2008-06-08,2.0000000298023224
71782,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71782,PO19372114749,10-4020-000582,29485,1086,1086,CARGO TRANSPORT 5,null,33319.98617553711,3182.8264,994.6333,37497.445892333984,2008-06-08,0.0
71783,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71783,PO19343113609,10-4020-000024,29957,992,992,CARGO TRANSPORT 5,null,65683.36844013452,6708.6743,2096.4607,74488.50344990015,2008-06-08,0.33000000193715096
71784,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71784,PO19285135919,10-4020-000448,29736,659,659,CARGO TRANSPORT 5,null,89869.27634851422,8684.946,2714.046,101268.26853601422,2008-06-08,0.12999999895691872
71796,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71796,PO17052159664,10-4020-000420,29660,1058,1058,CARGO TRANSPORT 5,null,47848.02697753906,4610.7705,1440.8658,53899.663330078125,2008-06-08,0.0
71797,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71797,PO16501134889,10-4020-000142,29796,642,642,CARGO TRANSPORT 5,null,65123.463148807794,6242.375,1950.7422,73316.58033630779,2008-06-08,0.24000000208616257
71815,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71815,PO13021155785,10-4020-000276,30089,1034,1034,CARGO TRANSPORT 5,null,926.9160213470459,91.3263,28.5395,1046.7818222045898,2008-06-08,0.0
71816,2,2008-06-01,2008-06-13,2008-06-08,5,False,SO71816,PO12992180445,10-4020-000295,30027,1038,1038,CARGO TRANSPORT 5,null,2847.408000946045,271.8533,84.9541,3204.215404510498,2008-06-08,0.0


In [0]:
(
    orders.write.format("delta")
    .mode("append") 
    .option(
        "path",
        "abfss://gold@intechaccstorage.dfs.core.windows.net/fact_sales",
    )
    .saveAsTable("databricksintech.gold.fact_sales")
)

In [0]:
# Reference target Delta table
gold_table = DeltaTable.forName(spark, "databricksintech.gold.fact_sales")

# Execute the Merge (Upsert)
gold_table.alias("target").merge(
    source = orders.alias("source"),
    condition = (
        "target.Sales_order_id = source.Sales_order_id"
        
    )
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Done!")

Done!
